
---

### Declaration of Usage of Generative AI
In this work, Generative AI was used to post-edit the wording of **some** written paragraphs and comments, including the ones in the final report, to improve their clarity and style, as English is not the native language of any of the group members. The underlying ideas and content remain entirely our own. In terms of coding, AI assistance was used **solely** for debugging purposes when exceptions occurred in the code. **No** generative AI tools were used as a learning assistant during the completion of this exercise.

---



### Source: [link](https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html#exercise-computing-word-embeddings-continuous-bag-of-words)

# Word Embeddings: Encoding Lexical Semantics

Word embeddings are dense vectors of real numbers, one per word in your
vocabulary. In NLP, it is almost always the case that your features are
words! But how should you represent a word in a computer? You could
store its ascii character representation, but that only tells you what
the word *is*, it doesn't say much about what it *means* (you might be
able to derive its part of speech from its affixes, or properties from
its capitalization, but not much). Even more, in what sense could you
combine these representations? We often want dense outputs from our
neural networks, where the inputs are $|V|$ dimensional, where
$V$ is our vocabulary, but often the outputs are only a few
dimensional (if we are only predicting a handful of labels, for
instance). How do we get from a massive dimensional space to a smaller
dimensional space?

How about instead of ascii representations, we use a one-hot encoding?
That is, we represent the word $w$ by

\begin{align}\overbrace{\left[ 0, 0, \dots, 1, \dots, 0, 0 \right]}^\text{|V| elements}\end{align}

where the 1 is in a location unique to $w$. Any other word will
have a 1 in some other location, and a 0 everywhere else.

There is an enormous drawback to this representation, besides just how
huge it is. It basically treats all words as independent entities with
no relation to each other. What we really want is some notion of
*similarity* between words. Why? Let's see an example.

Suppose we are building a language model. Suppose we have seen the
sentences

* The mathematician ran to the store.
* The physicist ran to the store.
* The mathematician solved the open problem.

in our training data. Now suppose we get a new sentence never before
seen in our training data:

* The physicist solved the open problem.

Our language model might do OK on this sentence, but wouldn't it be much
better if we could use the following two facts:

* We have seen  mathematician and physicist in the same role in a sentence. Somehow they
  have a semantic relation.
* We have seen mathematician in the same role  in this new unseen sentence
  as we are now seeing physicist.

and then infer that physicist is actually a good fit in the new unseen
sentence? This is what we mean by a notion of similarity: we mean
*semantic similarity*, not simply having similar orthographic
representations. It is a technique to combat the sparsity of linguistic
data, by connecting the dots between what we have seen and what we
haven't. This example of course relies on a fundamental linguistic
assumption: that words appearing in similar contexts are related to each
other semantically. This is called the `distributional
hypothesis <https://en.wikipedia.org/wiki/Distributional_semantics>`__.


# Getting Dense Word Embeddings

How can we solve this problem? That is, how could we actually encode
semantic similarity in words? Maybe we think up some semantic
attributes. For example, we see that both mathematicians and physicists
can run, so maybe we give these words a high score for the "is able to
run" semantic attribute. Think of some other attributes, and imagine
what you might score some common words on those attributes.

If each attribute is a dimension, then we might give each word a vector,
like this:

\begin{align}q_\text{mathematician} = \left[ \overbrace{2.3}^\text{can run},
   \overbrace{9.4}^\text{likes coffee}, \overbrace{-5.5}^\text{majored in Physics}, \dots \right]\end{align}

\begin{align}q_\text{physicist} = \left[ \overbrace{2.5}^\text{can run},
   \overbrace{9.1}^\text{likes coffee}, \overbrace{6.4}^\text{majored in Physics}, \dots \right]\end{align}

Then we can get a measure of similarity between these words by doing:

\begin{align}\text{Similarity}(\text{physicist}, \text{mathematician}) = q_\text{physicist} \cdot q_\text{mathematician}\end{align}

Although it is more common to normalize by the lengths:

\begin{align}\text{Similarity}(\text{physicist}, \text{mathematician}) = \frac{q_\text{physicist} \cdot q_\text{mathematician}}
   {\| q_\text{\physicist} \| \| q_\text{mathematician} \|} = \cos (\phi)\end{align}

Where $\phi$ is the angle between the two vectors. That way,
extremely similar words (words whose embeddings point in the same
direction) will have similarity 1. Extremely dissimilar words should
have similarity -1.


You can think of the sparse one-hot vectors from the beginning of this
section as a special case of these new vectors we have defined, where
each word basically has similarity 0, and we gave each word some unique
semantic attribute. These new vectors are *dense*, which is to say their
entries are (typically) non-zero.

But these new vectors are a big pain: you could think of thousands of
different semantic attributes that might be relevant to determining
similarity, and how on earth would you set the values of the different
attributes? Central to the idea of deep learning is that the neural
network learns representations of the features, rather than requiring
the programmer to design them herself. So why not just let the word
embeddings be parameters in our model, and then be updated during
training? This is exactly what we will do. We will have some *latent
semantic attributes* that the network can, in principle, learn. Note
that the word embeddings will probably not be interpretable. That is,
although with our hand-crafted vectors above we can see that
mathematicians and physicists are similar in that they both like coffee,
if we allow a neural network to learn the embeddings and see that both
mathematicians and physicists have a large value in the second
dimension, it is not clear what that means. They are similar in some
latent semantic dimension, but this probably has no interpretation to
us.


In summary, **word embeddings are a representation of the *semantics* of
a word, efficiently encoding semantic information that might be relevant
to the task at hand**. You can embed other things too: part of speech
tags, parse trees, anything! The idea of feature embeddings is central
to the field.


# Word Embeddings in Pytorch

Before we get to a worked example and an exercise, a few quick notes
about how to use embeddings in Pytorch and in deep learning programming
in general. Similar to how we defined a unique index for each word when
making one-hot vectors, we also need to define an index for each word
when using embeddings. These will be keys into a lookup table. That is,
embeddings are stored as a $|V| \times D$ matrix, where $D$
is the dimensionality of the embeddings, such that the word assigned
index $i$ has its embedding stored in the $i$'th row of the
matrix. In all of my code, the mapping from words to indices is a
dictionary named word\_to\_ix.

The module that allows you to use embeddings is torch.nn.Embedding,
which takes two arguments: the vocabulary size, and the dimensionality
of the embeddings.

To index into this table, you must use torch.LongTensor (since the
indices are integers, not floats).




In [ ]:
# Author: Robert Guthrie

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
torch.manual_seed(1)

from typing import Set, List, Tuple, Mapping

In [ ]:
import os
import random
import torch
import numpy as np


def seed_everything(seed: int = 1) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    os.environ["PYTHONHASHSEED"] = str(seed)

    print(f"[Seeded everything with seed={seed}]")

seed_everything(1)


[Seeded everything with seed=1]


In [ ]:
word_to_ix = {"hello": 0, "world": 1}
embeds = nn.Embedding(2, 5)  # 2 words in vocab, 5 dimensional embeddings
lookup_tensor = torch.tensor([word_to_ix["hello"]], dtype=torch.long)
hello_embed = embeds(lookup_tensor)
print(hello_embed)

tensor([[ 0.6614,  0.2669,  0.0617,  0.6213, -0.4519]],
       grad_fn=<EmbeddingBackward0>)


# An Example: N-Gram Language Modeling

Recall that in an n-gram language model, given a sequence of words
$w$, we want to compute

\begin{align}P(w_i | w_{i-1}, w_{i-2}, \dots, w_{i-n+1} )\end{align}

Where $w_i$ is the ith word of the sequence.

In this example, we will compute the loss function on some training
examples and update the parameters with backpropagation.

In [ ]:
CONTEXT_SIZE = 2
EMBEDDING_DIM = 10
# We will use Shakespeare Sonnet 2
test_sentence = """When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a totter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.""".split()
# we should tokenize the input, but we will ignore that for now
# build a list of tuples.  Each tuple is ([ word_i-2, word_i-1 ], target word)
trigrams = [([test_sentence[i], test_sentence[i + 1]], test_sentence[i + 2])
            for i in range(len(test_sentence) - 2)]
# print the first 3, just so you can see what they look like
print(trigrams[:3])

vocab = set(test_sentence)
word_to_ix = {word: i for i, word in enumerate(vocab)}


class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        # Reshape the embedded input to (batch_size, context_size * embedding_dim)
        embeds = embeds.view(embeds.size(0), -1)
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs


losses = []
loss_function = nn.NLLLoss() # Negative Log Likelihood Loss
model = NGramLanguageModeler(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for context, target in trigrams:

        # Step 1. Prepare the inputs to be passed to the model (i.e, turn the words
        # into integer indices and wrap them in tensors)
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)

        # Step 2. Recall that torch *accumulates* gradients. Before passing in a
        # new instance, you need to zero out the gradients from the old
        # instance
        model.zero_grad()

        # Step 3. Run the forward pass, getting log probabilities over next
        # words
        log_probs = model(context_idxs.unsqueeze(0)) # Add batch dimension

        # Step 4. Compute your loss function. (Again, Torch wants the target
        # word wrapped in a tensor)
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))

        # Step 5. Do the backward pass and update the gradient
        loss.backward()
        optimizer.step()

        # Get the Python number from a 1-element Tensor by calling tensor.item()
        total_loss += loss.item()
    print("Loss in Epoch {ep}: {l}".format(ep=epoch, l=np.round(total_loss, 2))) # The loss decreased every iteration over the training data!
    losses.append(total_loss)

[(['When', 'forty'], 'winters'), (['forty', 'winters'], 'shall'), (['winters', 'shall'], 'besiege')]
Loss in Epoch 0: 520.7
Loss in Epoch 1: 518.14
Loss in Epoch 2: 515.59
Loss in Epoch 3: 513.06
Loss in Epoch 4: 510.55
Loss in Epoch 5: 508.05
Loss in Epoch 6: 505.57
Loss in Epoch 7: 503.1
Loss in Epoch 8: 500.64
Loss in Epoch 9: 498.2


# Exercise: Computing Word Embeddings: Continuous Bag-of-Words

The Continuous Bag-of-Words model (CBOW) is frequently used in NLP deep
learning. It is a model that tries to predict words given the context of
a few words before and a few words after the target word. This is
distinct from language modeling, since CBOW is not sequential and does
not have to be probabilistic. Typcially, CBOW is used to quickly train
word embeddings, and these embeddings are used to initialize the
embeddings of some more complicated model. Usually, this is referred to
as *pretraining embeddings*. It almost always helps performance a couple
of percent.

The CBOW model is as follows. Given a target word $w_i$ and an
$N$ context window on each side, $w_{i-1}, \dots, w_{i-N}$
and $w_{i+1}, \dots, w_{i+N}$, referring to all context words
collectively as $C$, CBOW tries to minimize

\begin{align}-\log p(w_i | C) = -\log \text{Softmax}(A(\sum_{w \in C} q_w) + b)\end{align}

where $q_w$ is the embedding for word $w$.


## Exercise Layout
### 1. <u>Training CBOW Embeddings</u>
1.1) Implement a CBOW Model by completing ```class CBOW(nn.Module)``` and train it on ```raw_text```.    

1.2) Load Datasets ```tripadvisor_hotel_reviews_reduced.csv``` and ```scifi_reduced.txt```.     

1.3) Decide preprocessing steps by completing the function ```def custom_preprocess()```. Describe your decisions. Note that it's your choice to create different preprocessing functions for hotel reviews and scifi datasets or use the same preprocessing function.             

1.4) Train CBOW2 with a context width of 2 (in both directions) for the Hotel Reviews dataset.   

1.5) Train CBOW5 with a context width of 5 (in both directions) for the Hotel Reviews dataset. Are predictions made by the model sensitive towards the context size?
     
1.6) Train CBOW2 with a context width of 2 (in both directions) for the Sci-Fi story dataset.  


### 2. <u>Test your Embeddings</u>
Note - Do the following for CBOW2, and optionally for CBOW5

2.1) For the hotel reviews dataset, choose 3 nouns, 3 verbs, and 3 adjectives. Make sure that some nouns/verbs/adjectives occur frequently in the corpus and that others are rare. For each of the 9 chosen words, retrieve the 5 closest words according to your trained CBOW2 model. List them in your report and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   

2.2) Do the same for Sci-Fi dataset.   

2.3) How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.   

2.4) Choose 2 words and retrieve their 5 closest neighbours according to hotel review-based embeddings and the Sci-fi-based embeddings. Do they have different neighbours? If yes, can you reason why?    

2.5) What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?   

2.6) Load the pretrained embedding model with the given code snippet and retrieve the 5 closest neighbours using the embeddings from this pretrained model for your selection of words. Compare the hereby retrieved neighbours with the ones you retrieved above. Are there any similarities? How do they differ? Can you give judgment about the embedding quality of this pretrained model?


### Tips

1. Switch from CPU to a GPU instance after you have confirmed that your training procedure is working correctly.
2. You can always save your intermediate results (embeddings, preprocessed dataset, model, etc.) in your google drive via colab



### 1.1 Create a CBOW Model by completing ```class CBOW(nn.Module)``` and test it on ```raw_text```
Implement CBOW in Pytorch by filling in the class below. Some
tips:

* Think about which parameters you need to define.
* Make sure you know what shape each operation expects. Use .view() if you need to
  reshape.

In [ ]:
CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right
raw_text = """We are about to study the idea of a computational process.
Computational processes are abstract beings that inhabit computers.
As they evolve, processes manipulate other abstract things called data.
The evolution of a process is directed by a pattern of rules
called a program. People create programs to direct processes. In effect,
we conjure the spirits of the computer with our spells.""".split()

# By deriving a set from `raw_text`, we deduplicate the array
vocab = set(raw_text)
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}
data = []
for i in range(2, len(raw_text) - 2):
    context = [raw_text[i - 2], raw_text[i - 1],
               raw_text[i + 1], raw_text[i + 2]]
    target = raw_text[i]
    data.append((context, target))
print(data[:5])


# Create a CBOW class that predicts the target word from the given context.
# The neural network is shallow, i.e., it has one hidden layer.
class CBOW(nn.Module):

    def __init__(self,
                 vocab_size: int,
                 embedding_dim: int) -> None:

        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.proj = nn.Linear(embedding_dim, 32)   # self.proj layer takes the averaged context
        self.output = nn.Linear(32, vocab_size)    # embeddings as input (size = embedding_dim)

    def forward(self,
                inputs: torch.Tensor) -> torch.Tensor:
        embeds = torch.mean(self.embeddings(inputs), dim=1)  # Average the embeddings across the
        out = F.relu(self.proj(embeds))                      # context words (dimension 1)
        out = self.output(out)
        log_probs = F.log_softmax(out, dim=-1)
        return log_probs

[(['We', 'are', 'to', 'study'], 'about'), (['are', 'about', 'study', 'the'], 'to'), (['about', 'to', 'the', 'idea'], 'study'), (['to', 'study', 'idea', 'of'], 'the'), (['study', 'the', 'of', 'a'], 'idea')]


In [ ]:
# This function creates a context vector and safely handles unknown words using .get() with UNK fallback
def make_context_vector(context: List[str],
                        word_to_ix: Mapping[str, int]) -> torch.LongTensor:

    # Raise an exception if it does not contain '<UNK>' token
    if '<UNK>' not in word_to_ix:
      raise KeyError(f"Missing <UNK> in word_to_ix!")

    idxs = [word_to_ix.get(w, word_to_ix['<UNK>']) for w in context]
    return torch.tensor(idxs, dtype=torch.long)


# Add the unknown token to the vocabulary and word_to_ix dictionary
if '<UNK>' not in vocab:
    vocab.add('<UNK>')
    word_to_ix['<UNK>'] = len(word_to_ix)
    vocab_size = len(vocab)


make_context_vector(data[0][0], word_to_ix)  # example

tensor([32, 43, 14, 48])

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Define a custom CBOW Dataset that returns context word indices and the target word index
class CBOWDataset(Dataset):
    def __init__(self,
                 data: List[Tuple[List[str], str]],
                 word_to_ix: Mapping[str, int]) -> None:
        self.data = data
        self.word_to_ix = word_to_ix

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self,
                    idx: int) -> Tuple[List[int], int]:
        context, target = self.data[idx]
        context_idxs = make_context_vector(context, self.word_to_ix)
        target_idx = torch.tensor(self.word_to_ix.get(target, self.word_to_ix['<UNK>']),
                                  dtype=torch.long)
        return context_idxs, target_idx

In [ ]:
# Function to generate batches
def generate_batches(data: List[Tuple[List[str], str]],
                     word_to_ix: Mapping[str, int],
                     batch_size: int = 8) -> DataLoader:
    dataset = CBOWDataset(data, word_to_ix)
    dataloader = DataLoader(dataset,
                            batch_size=batch_size,
                            shuffle=True)
    return dataloader


#Pre-defined functions are not required after implementing MyDataset as functions already included in there
    # Function to get context vectors with batched data
    #def make_context_vector2(context_list, word_to_ix):
    #   batch_context_idxs = []
    #   for context in context_list:
    #     idxs = [word_to_ix[w] for w in context]
    #     batch_context_idxs.append(idxs)
    #   return torch.tensor(batch_context_idxs, dtype=torch.long)

    # Function to get context vectors with batched data
    #def make_labels_idx(labels, word_to_ix):
    #    label_idxs = [word_to_ix[w] for w in labels]
    #   return torch.tensor(label_idxs, dtype=torch.long)


# Function to train the model
# Instead of total loss, which results in a large number, we stick to tracking average loss
def train(model: nn.Module,
          data: List[Tuple[List[str], str]],
          device: torch.device,
          word_to_ix: Mapping[str, int] = word_to_ix,
          batch_size: int = 8,
          num_epochs: int = 15,
          lr: float = 1e-3) -> None:

    batches = generate_batches(data, word_to_ix, batch_size)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_function = nn.NLLLoss()

    model = model.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0

        for contexts, targets in batches:
          contexts = contexts.to(device)
          targets = targets.to(device)

          optimizer.zero_grad()
          log_probs = model(contexts)
          loss = loss_function(log_probs, targets)
          loss.backward()
          optimizer.step()

          total_loss += loss.item() * targets.size(0)
          total_examples += targets.size(0)

        avg_loss = total_loss / max(total_examples, 1)
        print(f"Epoch {epoch+1}/{num_epochs} | Avg Loss: {avg_loss:.4f}")

In [ ]:
# We initialize our model and train it. We observe the average loss
# decreasing over epochs, as should be if the implementation is correct.
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model_CBOW = CBOW(vocab_size, embedding_dim=10)
train(model_CBOW, data, device)

Epoch 1/15 | Avg Loss: 3.9248
Epoch 2/15 | Avg Loss: 3.8947
Epoch 3/15 | Avg Loss: 3.8700
Epoch 4/15 | Avg Loss: 3.8464
Epoch 5/15 | Avg Loss: 3.8224
Epoch 6/15 | Avg Loss: 3.7992
Epoch 7/15 | Avg Loss: 3.7747
Epoch 8/15 | Avg Loss: 3.7502
Epoch 9/15 | Avg Loss: 3.7241
Epoch 10/15 | Avg Loss: 3.6971
Epoch 11/15 | Avg Loss: 3.6695
Epoch 12/15 | Avg Loss: 3.6397
Epoch 13/15 | Avg Loss: 3.6104
Epoch 14/15 | Avg Loss: 3.5777
Epoch 15/15 | Avg Loss: 3.5470


### 1.2 Load Datasets

In [ ]:
import pandas as pd

In [ ]:
# Download Datasets tripadvisor_hotel_reviews_reduced.csv and scifi_reduced.txt
!mkdir "content"
!gdown "https://drive.google.com/uc?id=1foE1JuZJeu5E_4qVge9kExzhvF32teuF" -O "content/tripadvisor_hotel_reviews_reduced.csv" # For Hotel Reviews
!gdown "https://drive.google.com/uc?id=13IWXrTjGTrfCd9l7dScZVO8ZvMicPU75" -O "content/scifi_reduced.txt"  # For Scifi-Text

Downloading...
From: https://drive.google.com/uc?id=1foE1JuZJeu5E_4qVge9kExzhvF32teuF
To: /kaggle/working/content/tripadvisor_hotel_reviews_reduced.csv
100%|███████████████████████████████████████| 7.36M/7.36M [00:00<00:00, 275MB/s]
Downloading...
From: https://drive.google.com/uc?id=13IWXrTjGTrfCd9l7dScZVO8ZvMicPU75
To: /kaggle/working/content/scifi_reduced.txt
100%|███████████████████████████████████████| 43.1M/43.1M [00:00<00:00, 238MB/s]



### ⚠️ **Please, change the path if you are working locally or in Colab**


In [ ]:
#Load the datasets
reviews = pd.read_csv('/kaggle/working/content/tripadvisor_hotel_reviews_reduced.csv')
with open(f'/kaggle/working/content/scifi_reduced.txt') as f:
    scifi = f.read().splitlines()
scifi = pd.DataFrame({'text': scifi})

### 1.3 Preprocess Datasets

🗒❓ Describe your decisions for preprocessing the datasets

 ✅ For both the Hotel Reviews and Sci-Fi Text datasets, we applied a consistent preprocessing pipeline to clean and normalize the text before training the CBOW model. The goal was to remove noise, ensure consistency, and reduce vocabulary size without losing meaningful context.

In particular, we first converted all text to lowercase and stripped leading and trailing whitespace to make the vocabulary case-insensitive. Next, punctuation marks, digits and other non-alphabetic characters were replaced with spaces, leaving only letters and whitespace. This step helps remove elements that generally don't carry semantic meaning. Finally, the cleaned text was then tokenized using NLTK's word_tokenize(), which splits the text into individual word tokens while automatically handling multiple spaces and line breaks.

In [ ]:
# Import libraries for preprocessing
import re
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# Complete the preprocessing function and apply it to the datasets
import string
def custom_preprocess(raw_text: str) -> str:

    if raw_text is None:
      return ''

    # Lowercase, remove leading and trailing whitespaces
    text = str(raw_text).lower().strip()
    if not text:
      return ''

    # Remove digits and punctuation
    text = ''.join(ch if (ch.isalpha() or ch.isspace()) else ' ' for ch in text)

    # Tokenize, while collapsing multiple spaces
    tokens = word_tokenize(text)

    return tokens

# Preprocess the downloaded datasets
reviews['cleaned'] = reviews['Review'].apply(custom_preprocess)
scifi['cleaned'] = scifi['text'].apply(custom_preprocess)

In [ ]:
# Function to get vocab
def get_vocab(raw_llist: List[List[str]]) -> Set[str]:

  # Get unique words from the text
  vocab = {tok for tokens in raw_llist for tok in tokens}

  # Add '<UNK>' token
  if '<UNK>' not in vocab:
    vocab.add('<UNK>')

  return vocab

In [ ]:
# Function to get word-to-ix dictionary
def get_word2ix(vocab: Set[str]) -> Mapping[str, int]:
    return {word: i for i, word in enumerate(sorted(vocab))}

In [ ]:
# Function to generate tuples of context-target
def get_data(raw_llist: List[List[str]],
             context_window: int) -> List[Tuple[List[str], str]]:
    if context_window < 1:
        raise ValueError("context_window must be >= 1")

    samples = []
    for tokens in raw_llist:
        n = len(tokens)

        # No valid center positions if the text is too short
        if n < 2 * context_window + 1:
          continue

        for i in range(context_window, n - context_window):
            left_ctx = tokens[i - context_window: i]
            right_ctx = tokens[i + 1: i + context_window + 1]
            context = left_ctx + right_ctx
            target = tokens[i]
            samples.append((context, target))

    return samples

### 1.4 Train CBOW2 with a context width of 2 (in both directions) for the Hotel Reviews dataset.

In [ ]:
# Build the vocabulary from all tokenized reviews and compute its size
reviews_vocab = get_vocab(reviews['cleaned'])
reviews_vocab_size = len(reviews_vocab)

# Create a word-to-index mapping
reviews_word2ix = get_word2ix(reviews_vocab)

# Generate (context, target) training pairs for context of size 2
reviews_data2 = get_data(reviews['cleaned'], 2)

In [ ]:
# Train CBOW2 on the Hotel Reviews dataset using embedding_dim=50 and batch_size=128
# as provided in the task
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
reviews_CBOW2 = CBOW(reviews_vocab_size, embedding_dim=50)
train(reviews_CBOW2, reviews_data2, device, reviews_word2ix, batch_size=128, num_epochs=15, lr=0.001)

Epoch 1/15 | Avg Loss: 7.5465
Epoch 2/15 | Avg Loss: 7.0472
Epoch 3/15 | Avg Loss: 6.8129
Epoch 4/15 | Avg Loss: 6.6551
Epoch 5/15 | Avg Loss: 6.5366
Epoch 6/15 | Avg Loss: 6.4429
Epoch 7/15 | Avg Loss: 6.3656
Epoch 8/15 | Avg Loss: 6.3000
Epoch 9/15 | Avg Loss: 6.2432
Epoch 10/15 | Avg Loss: 6.1938
Epoch 11/15 | Avg Loss: 6.1494
Epoch 12/15 | Avg Loss: 6.1102
Epoch 13/15 | Avg Loss: 6.0747
Epoch 14/15 | Avg Loss: 6.0428
Epoch 15/15 | Avg Loss: 6.0137


### 1.5 Train CBOW5 with a context width of 5 (in both directions) for the Hotel Reviews dataset.  

In [ ]:
# Generate (context, target) training pairs for context of size 5
reviews_data5 = get_data(reviews['cleaned'], 5)

In [ ]:
# Train CBOW5 on the Hotel Reviews dataset using embedding_dim=50 and batch_size=128
# as provided in the task
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
reviews_CBOW5 = CBOW(reviews_vocab_size, embedding_dim=50)
train(reviews_CBOW5, reviews_data5, device, reviews_word2ix, batch_size=128, num_epochs=15, lr=0.001)

Epoch 1/15 | Avg Loss: 7.6214
Epoch 2/15 | Avg Loss: 7.1515
Epoch 3/15 | Avg Loss: 6.9430
Epoch 4/15 | Avg Loss: 6.8051
Epoch 5/15 | Avg Loss: 6.6999
Epoch 6/15 | Avg Loss: 6.6142
Epoch 7/15 | Avg Loss: 6.5405
Epoch 8/15 | Avg Loss: 6.4771
Epoch 9/15 | Avg Loss: 6.4209
Epoch 10/15 | Avg Loss: 6.3708
Epoch 11/15 | Avg Loss: 6.3253
Epoch 12/15 | Avg Loss: 6.2843
Epoch 13/15 | Avg Loss: 6.2466
Epoch 14/15 | Avg Loss: 6.2122
Epoch 15/15 | Avg Loss: 6.1798


🗒❓ Are predictions made by the model sensitive towards the context size?

✅ Yes, predictions made by CBOW model are sensitive to the context size. A larger context window allows the model to capture broader semantic relationships by including more surrounding words, which can improve generalization but may introduce noise. In contrast, a smaller context size makes the model focus on more specific and local word relationships, often improving precision but reducing its ability to understand wider context.

### 1.6 Train CBOW2 with a context width of 2 (in both directions) for the Sci-Fi story dataset

In [ ]:
# Build the vocabulary from tokenized scifi story and compute its size
scifi_vocab = get_vocab(scifi['cleaned'])
scifi_vocab_size = len(scifi_vocab)

# Create a word-to-index mapping
scifi_word2ix = get_word2ix(scifi_vocab)

# Generate (context, target) training pairs for context of size 2
scifi_data2 = get_data(scifi['cleaned'], 2)

In [ ]:
# Train CBOW2 on the Sci-Fi story dataset using embedding_dim=50 and batch_size=128
# as provided in the task
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
scifi_CBOW2 = CBOW(scifi_vocab_size, embedding_dim=50)
train(scifi_CBOW2, scifi_data2, device, scifi_word2ix, batch_size=128, num_epochs=3, lr=0.001)

Epoch 1/3 | Avg Loss: 6.6982
Epoch 2/3 | Avg Loss: 6.5163
Epoch 3/3 | Avg Loss: 6.4775


### 2.1 For the hotel reviews dataset, choose 3 nouns, 3 verbs, and 3 adjectives. (CBOW2 and optionally for CBOW5)
Make sure that some nouns/verbs/adjectives occur frequently in the corpus and that others are rare. For each of the 9 chosen words, retrieve the 5 closest words according to your trained CBOW2 model.

In [ ]:
# Retrieves the top-n most similar words to a given query word based on the model's
# learned embeddings. We did not adapt the function from the PDF in this case
# because it used nn.PairwiseDistance() which is Euclidean. However, from the lectures
# and the task above we know that the cosine similarity must be used to determine
# if the words are "close" to each other or "far" from each other.
def get_closest_word(model: nn.Module,
                     query_word: str,
                     word_to_ix: Mapping[str, int],
                     topn: int = 5) -> List[Tuple[str, float]]:

    if query_word not in word_to_ix:
        raise KeyError(f"'{query_word}' not in vocabulary")

    # Reverse mapping: index -> token
    ix_to_word = {idx: tok for tok, idx in word_to_ix.items()}

    # Ensure tensors are on the same device
    device = next(model.parameters()).device

    # Define the index of the query word
    query_ix = word_to_ix[query_word]

    with torch.no_grad():
        # (vocab_size, embedding_dim) embedding matrix
        embed_mat = model.embeddings.weight.to(device)

        # (embedding_dim,) vector for the query word
        query_vec = embed_mat[query_ix]

        # Cosine similarity with all tokens: shape (vocab_size,)
        sims = F.cosine_similarity(query_vec.unsqueeze(0), embed_mat, dim=1)

        # Exclude the query word itself
        sims[query_ix] = -1.0

        # Take top-k most similar
        k = min(topn, embed_mat.size(0) - 1)
        topk_values, topk_idxs = torch.topk(sims, k, largest=True, sorted=True)

    # Build (word, similarity) pairs
    return [(ix_to_word[int(j)], float(s)) for j, s in zip(topk_idxs, topk_values)]

# Test the implementation by printing the closest words to 'hotel' returned by review CBOW2
print(get_closest_word(reviews_CBOW2, 'hotel', reviews_word2ix))

[('interrior', 0.5618060827255249), ('jeje', 0.5613787174224854), ('vactation', 0.5496941804885864), ('gd', 0.5343438386917114), ('super', 0.5218315720558167)]


In [ ]:
# The function to create a dataframe with the word neighbours returned by the model
def neighbors_df(model: nn.Module,
                 model_name: str,
                 chosen_words: Mapping[str, List[str]],
                 word_to_ix: Mapping[str, int],
                 topn: int = 5) -> pd.DataFrame:
    cols = {}

    # Flatten words but keep input order
    words = [w for _, ws in chosen_words.items() for w in ws]

    for w in words:
        results = get_closest_word(model, w, word_to_ix, topn=topn)
        cols[w] = [f"{tok} ({sim:.3f})" for tok, sim in results]

    # Create a dataframe with the results, the first column is the name of the model
    df = pd.DataFrame(cols, index=[f"top_{i}" for i in range(1, topn + 1)])
    df.insert(0, "model", model_name)

    return df

# The function to visualize the results of two models at once for comparison
def compare_models(chosen_words: Mapping[str, List[str]],
                   model_CBOW2: nn.Module,
                   model_CBOW5: nn.Module,
                   word_to_ix: Mapping[str, int],
                   topn: int = 5) -> pd.DataFrame:

    # Build a dataframe for each model
    df2 = neighbors_df(model_CBOW2, "CBOW-2", chosen_words, word_to_ix, topn)
    df5 = neighbors_df(model_CBOW5, "CBOW-5", chosen_words, word_to_ix, topn)

    # Stack two dataframes
    stacked = pd.concat([df2, df5], axis=0)
    stacked.index = pd.MultiIndex.from_arrays([stacked["model"].values, stacked.index],
                                              names=["model", "rank"])
    stacked = stacked.drop(columns="model")

    return stacked

In [ ]:
# Visualize the neighbours returned by review CBOW2 and CBOW5 models
chosen_words = {
    "nouns": ["hotel", "room", "staff"],
    "verbs": ["stay", "enjoy", "recommend"],
    "adjectives": ["good", "bad", "friendly"]
}
compare_models(chosen_words, reviews_CBOW2, reviews_CBOW5, reviews_word2ix)

hotel                   room  \
model  rank                                                  
CBOW-2 top_1      interrior (0.562)          rooms (0.713)   
       top_2           jeje (0.561)            hug (0.531)   
       top_3      vactation (0.550)           crib (0.526)   
       top_4             gd (0.534)         hassel (0.523)   
       top_5          super (0.522)  transfersthis (0.513)   
CBOW-5 top_1      superably (0.561)      precisely (0.598)   
       top_2          trias (0.551)          terry (0.581)   
       top_3        bourbon (0.526)     tolietries (0.558)   
       top_4   consistently (0.519)         shoudl (0.554)   
       top_5  intelligently (0.519)    kitchenette (0.540)   

                           staff                    stay  \
model  rank                                                
CBOW-2 top_1    cleansed (0.551)           okura (0.537)   
       top_2       statt (0.512)         rightly (0.536)   
       top_3     bussola (0.504)          respit (0.536)   
       top_4   personnel (0.501)          expand (0.536)   
       top_5  flawlessly (0.498)          famine (0.520)   
CBOW-5 top_1      valets (0.687)         raphael (0.536)   
       top_2   employees (0.614)        discover (0.499)   
       top_3       angie (0.571)         wanting (0.497)   
       top_4    dwelling (0.557)           weeks (0.495)   
       top_5    speeders (0.516)  ramshackleness (0.489)   

                               enjoy             recommend  \
model  rank                                                  
CBOW-2 top_1       timeframe (0.563)     reccomend (0.602)   
       top_2           rehab (0.529)        ashley (0.563)   
       top_3         finicky (0.529)     maximises (0.539)   
       top_4        mcdonald (0.510)        tulane (0.538)   
       top_5        seafresh (0.504)  credentialed (0.533)   
CBOW-5 top_1        tastiest (0.571)   recommended (0.638)   
       top_2           scarf (0.562)       promote (0.520)   
       top_3  helllllllloooo (0.532)        mirage (0.519)   
       top_4              sb (0.512)        gooked (0.519)   
       top_5            loaf (0.512)        staffl (0.515)   

                            good                bad              friendly  
model  rank                                                                
CBOW-2 top_1      decent (0.683)       rave (0.596)      amenable (0.552)  
       top_2    competes (0.674)        cam (0.557)  scandanavian (0.540)  
       top_3   poorthere (0.595)      count (0.543)      screming (0.533)  
       top_4   fantastic (0.560)   positive (0.527)        polite (0.523)  
       top_5      superb (0.548)  strengths (0.520)        qantas (0.523)  
CBOW-5 top_1  reasonable (0.555)    meriden (0.558)     courteous (0.601)  
       top_2      itsome (0.540)   browsing (0.534)       browsed (0.530)  
       top_3   enjoyable (0.528)      worry (0.528)   comfortplus (0.517)  
       top_4       worng (0.525)      issue (0.519)            gu (0.507)  
       top_5   diversity (0.520)        ken (0.509)      helpfull (0.500)

🗒❓ List them in your report (at the end of this notebook) and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   

✅ The table above presents the top nearest neighbors identified by the model for several target words:

* **Nouns:** "hotel", "room", "staff"
* **Verbs:** "stay", "enjoy", "recommend"
* **Adjectives:** "good", "bad", "friendly"

Overall, the model performs well, and most of the retrieved neighbors make intuitive sense. For example, in **CBOW2**, the closest words such as *room -> rooms* (plural form of the same word), *recommend -> reccomend* (the same word with a typo), and *friendly -> amenable* (synonyms) demonstrate meaningful semantic relationships. Similarly, in **CBOW5**, pairs like *staff -> valets* (referring to personnel in the hotel), *recommend -> recommended* (past form of the same word), and *friendly -> courteous* (synonyms) confirm that the model captures linguistic similarity.

Even when the top-ranked neighbor seems less relevant (such as *good -> decent* in CBOW2), closer synonyms like *fantastic* or *superb* still appear within the top-5 results, showing that the model retrieves related words beyond the top-1 neighbors.

The results also highlight that learned embeddings are **dataset-dependent**. For instance, *crib* is the 3rd close neighbor to *room* in CBOW2. While *crib* traditionally means a child’s bed, in slang it means an apartment or house, which shows how users used this term in online reviews.

### 2.2 Repeat 2.1 for SciFi Dataset


In [ ]:
# Visualize the neighbours returned by scifi CBOW2 model
chosen_words = {
    "nouns": ["ship", "planet", "robot"],
    "verbs": ["travel", "attack", "explore"],
    "adjectives": ["strange", "dark", "futuristic"]
}
neighbors_df(scifi_CBOW2, "CBOW-2", chosen_words, scifi_word2ix)

,model,ship,planet,robot,travel,attack,explore,strange,dark,futuristic
top_1,CBOW-2,viewports (0.708),sleeth (0.679),girl (0.585),defeat (0.560),sustenance (0.582),get (0.692),pity (0.607),sky (0.585),transfinite (0.572)
top_2,CBOW-2,gate (0.695),war (0.669),walls (0.582),release (0.559),cullenlandl (0.581),widen (0.679),surprising (0.593),liabilities (0.578),ije (0.549)
top_3,CBOW-2,sleeth (0.694),vagaries (0.669),rendition (0.561),privileged (0.557),imposition (0.578),rebuild (0.678),incautious (0.591),worriedlooking (0.574),perigees (0.539)
top_4,CBOW-2,consensus (0.655),purpose (0.650),humbling (0.559),confederacy (0.544),roandidn (0.573),fall (0.667),respite (0.590),corridors (0.569),comit (0.533)
top_5,CBOW-2,trigger (0.652),purser (0.632),baby (0.547),inventions (0.539),wladetzky (0.570),settle (0.648),yet (0.579),lnoon (0.569),undefinable (0.531)


🗒❓ List your findings for SciFi Dataset as well, similarly to 2.1

✅ The table above presents the top nearest neighbors identified by the model for several target words:

* **Nouns:** "ship", "planet", "robot"
* **Verbs:** "travel", "attack", "explore"
* **Adjectives:** "strange", "dark", "futuristic"

Here, the results appear less intuitive, most likely due to the model being trained for **only 3 epochs**. Still, some meaningful relations are already visible, for example, *ship -> viewports* makes sense in the sci-fi spacecraft context, while *futuristic -> transfinite* similarly conveys a sense of abstraction.

### 2.3 🗒❓ How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.

✅ The hotel review-based embeddings are derived from text that is consistent and standardized, allowing them to develop a coherent vocabulary and well-defined semantic relationships. Because the language in hotel reviews is about concrete experiences, such as service quality, cleanliness, and comfort, the embeddings capture these associations in a stable and interpretable way.

In contrast, the sci-fi-based embeddings are trained on text which has creativity, imaginative concepts, and unconventional terminology. This leads to a less structured semantic space, where co-occurrence patterns are irregular. Consequently, these embeddings tend to be weaker and less consistent compared to those derived from hotel reviews.

### 2.4 Choose 2 words and retrieve their 5 closest neighbours according to hotel review-based embeddings and the Sci-fi-based embeddings.

In [ ]:
words_to_compare = ["hotel", "good"]
for word in words_to_compare:
  print(f"\nWord: '{word}'")

  cbow2_review_results = get_closest_word(reviews_CBOW2, word, reviews_word2ix)
  cbow2_scifi_results = get_closest_word(scifi_CBOW2, word, scifi_word2ix)

  print("\n  Hotel Review Embeddings:")
  for w, sim in cbow2_review_results:
      print(f"  {w:<15} | similarity={sim:.4f} ")

  print("\n  Sci-Fi Embeddings:")
  for w, sim in cbow2_scifi_results:
      print(f"   {w:<15} | similarity={sim:.4f} ")


Word: 'hotel'

  Hotel Review Embeddings:
  interrior       | similarity=0.5618 
  jeje            | similarity=0.5614 
  vactation       | similarity=0.5497 
  gd              | similarity=0.5343 
  super           | similarity=0.5218 

  Sci-Fi Embeddings:
   mesa            | similarity=0.5883 
   untranscended   | similarity=0.5715 
   customer        | similarity=0.5715 
   percuss         | similarity=0.5601 
   prospect        | similarity=0.5591 

Word: 'good'

  Hotel Review Embeddings:
  decent          | similarity=0.6827 
  competes        | similarity=0.6741 
  poorthere       | similarity=0.5954 
  fantastic       | similarity=0.5600 
  superb          | similarity=0.5481 

  Sci-Fi Embeddings:
   nice            | similarity=0.6471 
   ukrainian       | similarity=0.6165 
   late            | similarity=0.5927 
   marvelous       | similarity=0.5480 
   long            | similarity=0.5469 


🗒❓ Do they have different neighbours? If yes, can you reason why?

✅ Yes, they have different neighbours because embeddings are **dataset-dependent**: they capture word meanings based on the contexts in which words appear. In hotel reviews, *good* often relates to experiences or quality (good service, good room), while in sci-fi text, it may describe personal traits (good hero, good soldier). Similarly, *hotel* co-occurs with a room description in reviews but in sci-fi appears in futuristic or fictional settings, leading to different semantics in two cases.

### 2.5 🗒❓ What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?    

✅ Implementation-wise, CBOW2 and CBOW5 differ only in the size of their context windows: 2 and 5 words on each side of the target word, respectively. However, this difference in context window size leads to notable variations in what they capture:

* **CBOW2** focuses on a *local context* (2 words before and after), making it effective at capturing **syntactic** relationships. For example, **grammatical variants** of the same word such as *room* and *rooms*, which often occur in similar local contexts, but differ in form (singular vs. plural). This is why such pairs were retrieved by CBOW2 but not by CBOW5 in the table above. CBOW2 also captures **context-independent synonyms** like *good* and *superb*.

* **CBOW5** employs a *large context* window (5 words before and after), enabling it to capture broader **semantic** relationships. For instance, this helps to model **polysemy**, as it accounts for how word's meaning shifts across diverse contexts (immediate neighbors may remain similar for different word meanings, but its broader context often changes).



### 3. Use the following code snippet to load a pretrained GloVe embedding model
GloVe (Global Vectors for Word Representation) (link to paper: https://aclanthology.org/D14-1162/) is a count-based model trained on very large text corpora (Wikipedia, Common Crawl, Twitter, etc.). Unlike CBOW, which learns to predict a target word from its local context, GloVe learns embeddings that capture global co-occurrence statistics of words across the entire corpus.

Each word in GloVe has one static vector, i.e. its embedding does not change depending on context.

Note: Loading pretrained GloVe embeddings does not require a GPU. This runs efficiently on CPU. Change your run time to CPU again to save GPU compute units.

In [ ]:
!pip install gensim
!pip install "numpy<=1.26.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 51.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.2 which is incompat

In [ ]:
import numpy as np
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip
!unzip -q glove.6B.zip -d glove/

--2025-10-22 16:25:57--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-10-22 16:25:57--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-10-22 16:25:58--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
# Choose model
glove_input_file = "glove/glove.6B.100d.txt"
word2vec_output_file = "glove/glove.6B.100d.word2vec.txt"

# Convert GloVe format → Word2Vec format
glove2word2vec(glove_input_file, word2vec_output_file)

# Load model (this may take ~1–2 min)
model = KeyedVectors.load_word2vec_format(word2vec_output_file, binary=False)
print(f"Loaded {len(model.key_to_index):,} word vectors.")

/tmp/ipykernel_37/2719571235.py:6: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, word2vec_output_file)


Loaded 400,000 word vectors.


In [ ]:
### Insert your word selection from above here
trip_nouns = ["hotel", "room", "staff"]
trip_verbs = ["stay", "enjoy", "recommend"]
trip_adj = ["good", "bad", "friendly"]

for words_to_check in [trip_nouns, trip_verbs, trip_adj]:
  for w in words_to_check:
      if w in model.key_to_index:
          print(f"\nNearest neighbours for '{w}' in pretrained GloVe:")
          for neighbor, sim in model.most_similar(w, topn=5):
              print(f"  {neighbor:>12s}   (cosine similarity = {sim:.3f})")
      else:
          print(f"\n'{w}' not in vocabulary.")


Nearest neighbours for 'hotel' in pretrained GloVe:
        hotels   (cosine similarity = 0.793)
    restaurant   (cosine similarity = 0.776)
     apartment   (cosine similarity = 0.736)
           inn   (cosine similarity = 0.728)
        resort   (cosine similarity = 0.714)

Nearest neighbours for 'room' in pretrained GloVe:
         rooms   (cosine similarity = 0.842)
         floor   (cosine similarity = 0.800)
       bedroom   (cosine similarity = 0.766)
          door   (cosine similarity = 0.754)
        inside   (cosine similarity = 0.750)

Nearest neighbours for 'staff' in pretrained GloVe:
     personnel   (cosine similarity = 0.778)
     employees   (cosine similarity = 0.728)
      officers   (cosine similarity = 0.700)
       officer   (cosine similarity = 0.700)
      staffers   (cosine similarity = 0.699)

Nearest neighbours for 'stay' in pretrained GloVe:
         leave   (cosine similarity = 0.877)
            go   (cosine similarity = 0.810)
       staying   (cosine 

### 3.1 🗒❓ Compare the hereby retrieved neighbours with the ones you retrieved above. Are there any similarities to the results above? How do they differ?

❓ A. Compare the GloVe neighbours with those produced by your CBOW2 (and optionally CBOW5) models.<br> ✅ There are many overlaps between CBOW and GloVe retrieval results, for example, *room -> rooms*, *staff -> personnel*, and *recommend -> recommended*.

❓ B. Do the GloVe neighbours seem more general or semantically broader?<br>
✅ GloVe neighbours are semantically broader. This is because GloVe is trained using global co-occurrence statistics across the entire corpus, while CBOW focuses on predicting a target word based on its local context window.

❓ C. Are the CBOW neighbours more domain-specific to hotel reviews or the Sci-Fi text?<br> ✅ CBOW neighbours are domain-specific, reflecting the linguistic patterns and vocabulary of the training data. GloVe returns domain-independent pre-trained associations rather than hotel-specific or sci-fi ones since it was pre-trained on a large general text corpus. For instance, the neighbours *friendly -> cooperative* which is a more general or workplace-related rather than connected to the hotel review context.

❓ D. Discuss how training data size and training objective (predictive vs. count-based) might explain the observed differences.<br>
✅  GloVe was pre-trained on a large and diverse dataset, making its embeddings more general than our CBOW, trained on a relatively small and domain-specific coprus. The predictive objective in CBOW focuses on local context windows, producing narrower semantic relationships, whereas the count-based objective in GloVe leverages global co-occurrence statistics from the entire corpus, capturing broader contextual meaning.

## Report
### Abstract

This study investigates the use of Continuous Bag-of-Words (CBOW) models to generate word embeddings from two distinct textual domains: hotel reviews and science fiction literature. The objective is to analyze how context window size and dataset characteristics influence the semantic quality of the resulting embeddings.

### 1. Methodology

#### 1.1 Datasets
Two datasets were used for this study:

1. **Hotel Reviews**: A corpus of hotel reviews.
2. **Science Fiction Text**: A corpus of science fiction text.

#### 1.2 Text Preprocessing
Each dataset underwent preprocessing to ensure text uniformity and reduce noise. The steps included:
* Converting all text to lowercase
* Stripping leading and trailing whitespace
* Removing punctuation marks, digits, and non-alphabetic characters
* Tokenizing the text using NLTK’s `word_tokenize()`, which also handles multiple spaces and line breaks

#### 1.3 Dataset Construction
1. **Vocabulary Generation**
   * Extracted all unique words from the cleaned corpus using the `get_vocab()` function.
   * Constructed a word-to-index mapping with `get_word2ix()` to assign a unique integer ID to each word in the vocabulary.

2. **CBOW Pair Generation**
   * Created context–target pairs using `get_data()`, which generated word pairs based on the defined `context_window` size.

3. **Custom Dataset Class**
   * Implemented a `CBOWDataset` class inheriting from `torch.utils.data.Dataset`.
   * This class receives the CBOW pairs and vocabulary mapping to produce indexed context–target tensors.
   * The `__getitem__` method returns a tuple `(context_indices, target_index)` for each item.

4. **Dataloader Creation**
   * Using the dataset, obtained above, we created a dataloader within `generate_batches()` function to configure data batching and shuffling during training.

#### 1.4 Model Architecture
The CBOW model was implemented in PyTorch with the following components:
* **Embedding Layer**: Maps each word to a dense vector representation of dimension 50.
* **Hidden Layer**: Performs a linear transformation on averaged context embeddings (hidden dimension = 32).
* **ReLU Activation**: Introduced after the hidden layer to add non-linearity.
* **Output Layer**: Projects the model's internal representation to the vocabulary space, predicting the target word index.
* **Log Softmax**: Final activation function used to compute normalized log-probabilities, compatiable with the `nn.NLLLoss()` objective.

#### 1.5 Training Configuration

The following table shows the hyperparameters used in our experiments:

| **Parameter**           | **Hotel Reviews (CBOW2)** | **Hotel Reviews (CBOW5)** | **Sci-Fi (CBOW2)** |
| ----------------------- | ------------------------- | ------------------------- | ------------------ |
| **Embedding Dimension** | 50                        | 50                        | 50                 |
| **Context Window Size** | 2                         | 5                         | 2                  |
| **Optimizer**           | Adam                      | Adam                      | Adam               |
| **Learning Rate**       | 0.001                     | 0.001                     | 0.001              |
| **Number of Epochs**    | 15                        | 15                        | 3                  |
| **Batch Size**          | 128                       | 128                       | 128                |
| **Loss Function**       | NLLLoss                   | NLLLoss                   | NLLLoss            |

---

## Answers to the Questions

#### 🗒❓ Describe your decisions for preprocessing the datasets

✅ For both the Hotel Reviews and Sci-Fi Text datasets, we applied a consistent preprocessing pipeline to clean and normalize the text before training the CBOW model. The goal was to remove noise, ensure consistency, and reduce vocabulary size without losing meaningful context.

In particular, we first converted all text to lowercase and stripped leading and trailing whitespace to make the vocabulary case-insensitive. Next, punctuation marks, digits and other non-alphabetic characters were replaced with spaces, leaving only letters and whitespace. This step helps remove elements that generally don't carry semantic meaning. Finally, the cleaned text was then tokenized using NLTK's word_tokenize(), which splits the text into individual word tokens while automatically handling multiple spaces and line breaks.

<br>

#### 🗒❓ Are predictions made by the model sensitive towards the context size?

✅ Yes, predictions made by CBOW model are sensitive to the context size. A larger context window allows the model to capture broader semantic relationships by including more surrounding words, which can improve generalization but may introduce noise. In contrast, a smaller context size makes the model focus on more specific and local word relationships, often improving precision but reducing its ability to understand wider context.

<br>

#### 🗒❓ List them in your report (at the end of this notebook) and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   

| Model  | Rank  | hotel (score)         | room (score)          | staff (score)         | stay (score)          | enjoy (score)         | recommend (score)        | good (score)          | bad (score)           | friendly (score)        |
|---------|--------|----------------------|-----------------------|-----------------------|-----------------------|-----------------------|--------------------------|-----------------------|-----------------------|-------------------------|
| CBOW-2  | top_1  | interrior (0.562)    | rooms (0.713)         | cleansed (0.551)      | okura (0.537)         | timeframe (0.563)     | reccomend (0.602)        | decent (0.683)        | rave (0.596)          | amenable (0.552)        |
|         | top_2  | jeje (0.561)         | hug (0.531)           | statt (0.512)         | rightly (0.536)       | rehab (0.529)         | ashley (0.563)           | competes (0.674)      | cam (0.557)           | scandanavian (0.540)    |
|         | top_3  | vactation (0.550)    | crib (0.526)          | bussola (0.504)       | respit (0.536)        | finicky (0.529)       | maximises (0.539)        | poorthere (0.595)     | count (0.543)         | screming (0.533)        |
|         | top_4  | gd (0.534)           | hassel (0.523)        | personnel (0.501)     | expand (0.536)        | mcdonald (0.510)      | tulane (0.538)           | fantastic (0.560)     | positive (0.527)      | polite (0.523)          |
|         | top_5  | super (0.522)        | transfersthis (0.513) | flawlessly (0.498)    | famine (0.520)        | seafresh (0.504)      | credentialed (0.533)     | superb (0.548)        | strengths (0.520)     | qantas (0.523)          |
| CBOW-5  | top_1  | superably (0.561)    | precisely (0.598)     | valets (0.687)        | raphael (0.536)       | tastiest (0.571)      | recommended (0.638)      | reasonable (0.555)    | meriden (0.558)       | courteous (0.601)       |
|         | top_2  | trias (0.551)        | terry (0.581)         | employees (0.614)     | discover (0.499)      | scarf (0.562)         | promote (0.520)          | itsome (0.540)        | browsing (0.534)      | browsed (0.530)         |
|         | top_3  | bourbon (0.526)      | tolietries (0.558)    | angie (0.571)         | wanting (0.497)       | helllllllloooo (0.532)| mirage (0.519)           | enjoyable (0.528)     | worry (0.528)         | comfortplus (0.517)     |
|         | top_4  | consistently (0.519) | shoudl (0.554)        | dwelling (0.557)      | weeks (0.495)         | sb (0.512)            | gooked (0.519)           | worng (0.525)         | issue (0.519)         | gu (0.507)              |
|         | top_5  | intelligently (0.519)| kitchenette (0.540)   | speeders (0.516)      | ramshackleness (0.489)| loaf (0.512)          | staffl (0.515)           | diversity (0.520)     | ken (0.509)           | helpfull (0.500)        |

✅ The table above presents the top nearest neighbors identified by the model for several target words:

* **Nouns:** "hotel", "room", "staff"
* **Verbs:** "stay", "enjoy", "recommend"
* **Adjectives:** "good", "bad", "friendly"

Overall, the model performs well, and most of the retrieved neighbors make intuitive sense. For example, in **CBOW2**, the closest words such as *room -> rooms* (plural form of the same word), *recommend -> reccomend* (the same word with a typo), and *friendly -> amenable* (synonyms) demonstrate meaningful semantic relationships. Similarly, in **CBOW5**, pairs like *staff -> valets* (referring to personnel in the hotel), *recommend -> recommended* (past form of the same word), and *friendly -> courteous* (synonyms) confirm that the model captures linguistic similarity.

Even when the top-ranked neighbor seems less relevant (such as *good -> decent* in CBOW2), closer synonyms like *fantastic* or *superb* still appear within the top-5 results, showing that the model retrieves related words beyond the top-1 neighbors.

The results also highlight that learned embeddings are **dataset-dependent**. For instance, *crib* is the 3rd close neighbor to *room* in CBOW2. While *crib* traditionally means a child’s bed, in slang it means an apartment or house, which shows how users used this term in online reviews.

<br>

####  🗒❓ List your findings for SciFi Dataset as well, similarly to 2.1
| Model  | Rank  | ship (score)        | planet (score)       | robot (score)        | travel (score)       | attack (score)         | explore (score)       | strange (score)        | dark (score)          | futuristic (score)      |
|---------|--------|--------------------|----------------------|----------------------|----------------------|------------------------|-----------------------|------------------------|-----------------------|-------------------------|
| CBOW-2  | top_1  | viewports (0.708)  | sleeth (0.679)       | girl (0.585)         | defeat (0.560)       | sustenance (0.582)     | get (0.692)           | pity (0.607)           | sky (0.585)           | transfinite (0.572)     |
| CBOW-2  | top_2  | gate (0.695)       | war (0.669)          | walls (0.582)        | release (0.559)      | cullenlandl (0.581)    | widen (0.679)         | surprising (0.593)     | liabilities (0.578)   | ije (0.549)             |
| CBOW-2  | top_3  | sleeth (0.694)     | vagaries (0.669)     | rendition (0.561)    | privileged (0.557)   | imposition (0.578)     | rebuild (0.678)       | incautious (0.591)     | worriedlooking (0.574)| perigees (0.539)        |
| CBOW-2  | top_4  | consensus (0.655)  | purpose (0.650)      | humbling (0.559)     | confederacy (0.544)  | roandidn (0.573)       | fall (0.667)          | respite (0.590)        | corridors (0.569)     | comit (0.533)           |
| CBOW-2  | top_5  | trigger (0.652)    | purser (0.632)       | baby (0.547)         | inventions (0.539)   | wladetzky (0.570)      | settle (0.648)        | yet (0.579)            | lnoon (0.569)         | undefinable (0.531)     |

✅ The table above presents the top nearest neighbors identified by the model for several target words:

* **Nouns:** "ship", "planet", "robot"
* **Verbs:** "travel", "attack", "explore"
* **Adjectives:** "strange", "dark", "futuristic"

Here, the results appear less intuitive, most likely due to the model being trained for **only 3 epochs**. Still, some meaningful relations are already visible, for example, *ship -> viewports* makes sense in the sci-fi spacecraft context, while *futuristic -> transfinite* similarly conveys a sense of abstraction.

<br>

#### 🗒❓ How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.

✅ The hotel review-based embeddings are derived from text that is consistent and standardized, allowing them to develop a coherent vocabulary and well-defined semantic relationships. Because the language in hotel reviews is about concrete experiences, such as service quality, cleanliness, and comfort, the embeddings capture these associations in a stable and interpretable way.

In contrast, the sci-fi-based embeddings are trained on text which has creativity, imaginative concepts, and unconventional terminology. This leads to a less structured semantic space, where co-occurrence patterns are irregular. Consequently, these embeddings tend to be weaker and less consistent compared to those derived from hotel reviews.

<br>

#### 🗒❓ Do they have different neighbours? If yes, can you reason why?

✅ Yes, they have different neighbours because embeddings are **dataset-dependent**: they capture word meanings based on the contexts in which words appear. In hotel reviews, *good* often relates to experiences or quality (good service, good room), while in sci-fi text, it may describe personal traits (good hero, good soldier). Similarly, *hotel* co-occurs with a room description in reviews but in sci-fi appears in futuristic or fictional settings, leading to different semantics in two cases.

<br>

#### 🗒❓ What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?

✅ Implementation-wise, CBOW2 and CBOW5 differ only in the size of their context windows: 2 and 5 words on each side of the target word, respectively. However, this difference in context window size leads to notable variations in what they capture:

* **CBOW2** focuses on a *local context* (2 words before and after), making it effective at capturing **syntactic** relationships. For example, **grammatical variants** of the same word such as *room* and *rooms*, which often occur in similar local contexts, but differ in form (singular vs. plural). This is why such pairs were retrieved by CBOW2 but not by CBOW5 in the table above. CBOW2 also captures **context-independent synonyms** like *good* and *superb*.

* **CBOW5** employs a *large context* window (5 words before and after), enabling it to capture broader **semantic** relationships. For instance, this helps to model **polysemy**, as it accounts for how word's meaning shifts across diverse contexts (immediate neighbors may remain similar for different word meanings, but its broader context often changes).

<br>

#### 🗒❓ Compare the hereby retrieved neighbours with the ones you retrieved above. Are there any similarities to the results above? How do they differ?

❓ A. Compare the GloVe neighbours with those produced by your CBOW2 (and optionally CBOW5) models.<br> ✅ There are many overlaps between CBOW and GloVe retrieval results, for example, *room -> rooms*, *staff -> personnel*, and *recommend -> recommended*.

❓ B. Do the GloVe neighbours seem more general or semantically broader?<br>
✅ GloVe neighbours are semantically broader. This is because GloVe is trained using global co-occurrence statistics across the entire corpus, while CBOW focuses on predicting a target word based on its local context window.

❓ C. Are the CBOW neighbours more domain-specific to hotel reviews or the Sci-Fi text?<br> ✅ CBOW neighbours are domain-specific, reflecting the linguistic patterns and vocabulary of the training data. GloVe returns domain-independent pre-trained associations rather than hotel-specific or sci-fi ones since it was pre-trained on a large general text corpus. For instance, the neighbours *friendly -> cooperative* which is a more general or workplace-related rather than connected to the hotel review context.

❓ D. Discuss how training data size and training objective (predictive vs. count-based) might explain the observed differences.<br>
✅  GloVe was pre-trained on a large and diverse dataset, making its embeddings more general than our CBOW, trained on a relatively small and domain-specific coprus. The predictive objective in CBOW focuses on local context windows, producing narrower semantic relationships, whereas the count-based objective in GloVe leverages global co-occurrence statistics from the entire corpus, capturing broader contextual meaning.